# HLO 500 Scenario Execution and Provenance Audit
This notebook audits the supplied project, executes its structural 300+200 evaluation, and runs its separate resource recommender with saved parameters. It does not claim to recover the historical Table 5 ranking run. The supplied target-matched Excel trial values were previously constructed as numerical examples; matching their means does not establish an executed experiment. New rankings below are recorded independently of those targets.

## 1. Environment and project

In [1]:
from pathlib import Path
import csv, hashlib, importlib.util, json, platform, sys
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import rdflib
import yaml
from rdflib import Graph, RDF, RDFS, OWL, URIRef
from IPython.display import display

PROJECT_OVERRIDE = None
TOP_K = 5
RAW_ORIGINAL_LOG = None

def find_project():
    if PROJECT_OVERRIDE is not None:
        return Path(PROJECT_OVERRIDE).resolve()
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "HLO-main"):
            if (candidate / "evaluation/data/scenarios.csv").is_file() and (candidate / "src/hlo/recommendation/engine.py").is_file():
                return candidate
    raise FileNotFoundError("Set PROJECT_OVERRIDE to the extracted HLO-main directory")

PROJECT = find_project()
EVALUATION = PROJECT / "evaluation"
OUTPUT = EVALUATION / "execution_audit_500"
OUTPUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT / "src"))
METRICS = ["top1", "hit3", "hit5", "mrr", "map"]
print("Project:", PROJECT)
print("Output:", OUTPUT)
print("Python:", platform.python_version())

Project: /workspace/scratch/f00636c3be40/evaluation_original_audit/project/HLO-main
Output: /workspace/scratch/f00636c3be40/evaluation_original_audit/project/HLO-main/evaluation/execution_audit_500
Python: 3.12.14


## 2. Input paths and fingerprints

In [2]:
PATHS = {
    "before_owl": EVALUATION / "ontologies/before/HLO_T1.owl",
    "after_owl": EVALUATION / "ontologies/after/HLO_Template_LLM.owl",
    "parent_scenarios": EVALUATION / "data/parent_scenarios.csv",
    "reference_scenarios": EVALUATION / "data/reference_scenarios.csv",
    "structural_scenarios": EVALUATION / "data/scenarios.csv",
    "before_excel": EVALUATION / "results/HLO_500_before_overall_Details.xlsx",
    "after_excel": EVALUATION / "results/HLO_500_after_overall_Details.xlsx",
    "resources": PROJECT / "outputs/data/resources.csv",
    "recommendation_scenarios": PROJECT / "outputs/data/scenarios.csv",
    "recommendation_concepts": PROJECT / "outputs/data/concepts.csv",
    "config": PROJECT / "configs/default.yaml",
    "saved_summary": PROJECT / "outputs/reports/summary.json",
    "table5_summary": PROJECT / "src/hlo/evaluation/ontology_comparison_results.csv",
}

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

inventory = pd.DataFrame([
    {"role": role, "relative_path": str(path.relative_to(PROJECT)),
     "exists": path.is_file(), "bytes": path.stat().st_size if path.is_file() else None,
     "sha256": file_hash(path) if path.is_file() else None}
    for role, path in PATHS.items()
])
inventory.to_csv(OUTPUT / "input_inventory.csv", index=False)
display(inventory.drop(columns="sha256"))
missing = inventory.loc[~inventory["exists"], "role"].tolist()
if missing:
    raise FileNotFoundError(f"Missing project inputs: {missing}")

role,relative_path,exists,bytes
before_owl,evaluation/ontologies/before/HLO_T1.owl,True,985279
after_owl,evaluation/ontologies/after/HLO_Template_LLM.owl,True,1161174
parent_scenarios,evaluation/data/parent_scenarios.csv,True,42945
reference_scenarios,evaluation/data/reference_scenarios.csv,True,23805
structural_scenarios,evaluation/data/scenarios.csv,True,67152
before_excel,evaluation/results/HLO_500_before_overall_Details.xlsx,True,55443
after_excel,evaluation/results/HLO_500_after_overall_Details.xlsx,True,54619
resources,outputs/data/resources.csv,True,54154
recommendation_scenarios,outputs/data/scenarios.csv,True,25884
recommendation_concepts,outputs/data/concepts.csv,True,7978


## 3. Code roles

In [3]:
roles = pd.DataFrame([
    ("evaluation/generate_parent.py", "Select 300 shared MIMO parent cases", "selection_seed=73"),
    ("evaluation/generate_reference.py", "Select 200 distinct MIMO reference cases with an after code", "selection_seed=74"),
    ("evaluation/merge_scenarios.py", "Merge 300+200 structural cases", "no ranking trials"),
    ("evaluation/compare_ontologies.py", "Resolve identifiers and verify parents/codes", "structural correctness"),
    ("src/hlo/recommendation/engine.py", "Rank resource candidates using policy and weights", "deterministic; no seed"),
    ("src/hlo/evaluation/evaluator.py", "Evaluate the context scenario schema", "does not produce trial1/trial2"),
    ("src/hlo/data/generator.py", "Generate a synthetic resource/context corpus", "dataset seed; not Table 5 trial seeds"),
], columns=["file", "role", "randomness"])
roles["sha256"] = roles["file"].map(lambda p: file_hash(PROJECT / p))
roles.to_csv(OUTPUT / "code_roles.csv", index=False)
display(roles.drop(columns="sha256"))

file,role,randomness
evaluation/generate_parent.py,Select 300 shared MIMO parent cases,selection_seed=73
evaluation/generate_reference.py,Select 200 distinct MIMO reference cases with an after code,selection_seed=74
evaluation/merge_scenarios.py,Merge 300+200 structural cases,no ranking trials
evaluation/compare_ontologies.py,Resolve identifiers and verify parents/codes,structural correctness
src/hlo/recommendation/engine.py,Rank resource candidates using policy and weights,deterministic; no seed
src/hlo/evaluation/evaluator.py,Evaluate the context scenario schema,does not produce trial1/trial2
src/hlo/data/generator.py,Generate a synthetic resource/context corpus,dataset seed; not Table 5 trial seeds


## 4. Search for trial generation and seed evidence

In [4]:
needles = ("trial1", "trial2", "random.Random", "default_rng", "np.random", "random_state", "selection_seed")
source_hits = []
for path in sorted(PROJECT.rglob("*")):
    if not path.is_file() or path.suffix not in {".py", ".ipynb"}:
        continue
    if OUTPUT in path.parents:
        continue
    if path.suffix == ".ipynb":
        document = json.loads(path.read_text(encoding="utf-8"))
        text = "\n".join("".join(c.get("source", [])) for c in document.get("cells", []))
    else:
        text = path.read_text(encoding="utf-8", errors="replace")
    for number, line in enumerate(text.splitlines(), 1):
        found = [needle for needle in needles if needle in line]
        if found:
            source_hits.append({"file": str(path.relative_to(PROJECT)), "line_in_code_text": number,
                                "matches": " | ".join(found), "excerpt": line[:240]})
seed_evidence = pd.DataFrame(source_hits)
seed_evidence.to_csv(OUTPUT / "seed_and_trial_code_evidence.csv", index=False)
display(seed_evidence.head(30))

file,line_in_code_text,matches,excerpt
evaluation/generate_parent.py,73,random.Random,rng = random.Random(seed)
evaluation/generate_parent.py,87,selection_seed,"""selection_seed"": seed,"
evaluation/generate_reference.py,76,random.Random,rng = random.Random(seed)
evaluation/generate_reference.py,90,selection_seed,"""selection_seed"": seed,"
legacy/generator_hlo_long.py,130,np.random,"""relevance_score"": round(np.random.uniform(0.6, 0.99), 2)"
legacy/generator_hlo_long.py,187,np.random,"for t in types[:np.random.choice([1,2])]:"
legacy/generator_hlo_long.py,188,np.random,diff = np.random.choice(difficulties)
src/hlo/data/generator.py,44,random.Random,rng = random.Random(seed)
src/hlo/training/pipeline.py,63,random_state,"random_state=seed,"
src/hlo/training/pipeline.py,82,random_state,"n=min(100, len(train_scenarios)), random_state=seed"


## 5. OWL catalogs

In [5]:
def local(value):
    return str(value).rsplit("#", 1)[-1].rsplit("/", 1)[-1]

def owl_catalog(path):
    graph = Graph().parse(path)
    predicates = {local(p): p for p in set(graph.predicates())}
    records = []
    for subject in sorted(set(graph.subjects(RDF.type, OWL.Class)), key=str):
        def value(name):
            predicate = predicates.get(name)
            values = sorted(str(v) for v in graph.objects(subject, predicate)) if predicate else []
            return values[0] if values else ""
        records.append({"iri": str(subject), "class_local": local(subject),
                        "label": value("prefLabel") or value("label") or value("conceptName"),
                        "source_identifier": value("sourceIdentifier"),
                        "concept_code": value("conceptCode"), "source_ontology": value("sourceOntology"),
                        "category": value("conceptCategory"),
                        "parents": sorted(str(v) for v in graph.objects(subject, RDFS.subClassOf))})
    return graph, pd.DataFrame(records)

graphs, catalogs = {}, {}
for release in ("before", "after"):
    graphs[release], catalogs[release] = owl_catalog(PATHS[f"{release}_owl"])
    catalogs[release].assign(parents=lambda d: d.parents.map(json.dumps)).to_csv(OUTPUT / f"{release}_ontology_catalog.csv", index=False)
    print(release, "classes:", len(catalogs[release]), "triples:", len(graphs[release]))
display(catalogs["after"].groupby(["source_ontology", "category"], dropna=False).size().rename("classes").reset_index())

source_ontology,category,classes
,,32
AI-Enrichment,activity-context,12
AI-Enrichment,assessment,15
AI-Enrichment,clinical-skill,25
AI-Enrichment,competency,25
AI-Enrichment,educational-activity,18
AI-Enrichment,learning-objective,250
AI-Enrichment,learning-resource,12
AI-Enrichment,location-context,12
AI-Enrichment,recommendation,15


before classes: 1642 triples: 14651
after classes: 1642 triples: 17839


## 6. Ontology roots and ancestor paths

In [6]:
def ancestry(graph, subject):
    pending = [(subject, [local(subject)])]
    result = []
    while pending:
        node, path = pending.pop()
        parents = sorted(graph.objects(node, RDFS.subClassOf), key=str)
        if not parents:
            result.append(" -> ".join(path))
        for parent in parents:
            if local(parent) in path:
                result.append(" -> ".join(path + [local(parent), "cycle"]))
            else:
                pending.append((parent, path + [local(parent)]))
    return sorted(set(result))

root_rows = []
for release, graph in graphs.items():
    declared = set(graph.subjects(RDF.type, OWL.Class))
    for subject in sorted(declared, key=str):
        parents = list(graph.objects(subject, RDFS.subClassOf))
        if not parents or all(p == OWL.Thing or p not in declared for p in parents):
            root_rows.append({"release": release, "root": local(subject), "iri": str(subject),
                              "parents": json.dumps([str(p) for p in parents])})
roots = pd.DataFrame(root_rows)
roots.to_csv(OUTPUT / "ontology_roots.csv", index=False)
display(roots)

release,root,iri,parents
before,Context,http://example.org/hlo.owl#Context,"[""http://www.w3.org/2002/07/owl#Thing""]"
before,Learner,http://example.org/hlo.owl#Learner,"[""http://www.w3.org/2002/07/owl#Thing""]"
before,LearningResource,http://example.org/hlo.owl#LearningResource,"[""http://www.w3.org/2002/07/owl#Thing""]"
before,MedicalConcept,http://example.org/hlo.owl#MedicalConcept,"[""http://www.w3.org/2002/07/owl#Thing""]"
before,ResidentLevel,http://example.org/hlo.owl#ResidentLevel,"[""http://www.w3.org/2002/07/owl#Thing""]"
after,Context,http://example.org/hlo.owl#Context,"[""http://www.w3.org/2002/07/owl#Thing""]"
after,Learner,http://example.org/hlo.owl#Learner,"[""http://www.w3.org/2002/07/owl#Thing""]"
after,LearningResource,http://example.org/hlo.owl#LearningResource,"[""http://www.w3.org/2002/07/owl#Thing""]"
after,MedicalConcept,http://example.org/hlo.owl#MedicalConcept,"[""http://www.w3.org/2002/07/owl#Thing""]"
after,ResidentLevel,http://example.org/hlo.owl#ResidentLevel,"[""http://www.w3.org/2002/07/owl#Thing""]"


## 7. Load and validate the saved 300+200 scenarios

In [7]:
structural = pd.read_csv(PATHS["structural_scenarios"], keep_default_na=False)
assert len(structural) == 500
assert structural.scenario_id.is_unique and structural.source_identifier.is_unique
assert structural.scenario_type.value_counts().to_dict() == {"parent_placement": 300, "reference_lookup": 200}
assert set(structural.scenario_id) == {f"P{i:03d}" for i in range(1, 301)} | {f"R{i:03d}" for i in range(1, 201)}
display(structural.head(8))
display(structural.groupby(["scenario_type", "selection_seed"]).size().rename("scenarios").reset_index())

scenario_id,scenario_type,source_identifier,query,expected_label,expected_category,expected_parent,selection_seed,expected_concept_code
P001,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1,Alert,Alert,base-concept,MIMOConcept,73,
P002,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10,Biomedical Informatics,Biomedical Informatics,base-concept,MIMOConcept,73,
P003,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10001,data display,data display,base-concept,MIMOConcept,73,
P004,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10003,computer-aided design,computer-aided design,base-concept,MIMOConcept,73,
P005,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10012,workflow,workflow,base-concept,MIMOConcept,73,
P006,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10016,Agent-Based Modeling,Agent-Based Modeling,base-concept,MIMOConcept,73,
P007,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10021,probability theory,probability theory,base-concept,MIMOConcept,73,
P008,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10023,library associations,library associations,base-concept,MIMOConcept,73,


scenario_type,selection_seed,scenarios
parent_placement,73,300
reference_lookup,74,200


## 8. Scenario source resolution

In [8]:
resolution_rows = []
for release, catalog in catalogs.items():
    by_source = catalog[catalog.source_identifier.ne("")].set_index("source_identifier")
    for row in structural.to_dict("records"):
        resolved = row["source_identifier"] in by_source.index
        concept = by_source.loc[row["source_identifier"]] if resolved else None
        resolution_rows.append({"release": release, "scenario_id": row["scenario_id"],
            "source_identifier": row["source_identifier"], "resolved": resolved,
            "source_ontology": concept.source_ontology if resolved else "",
            "label": concept.label if resolved else "",
            "class_iri": concept.iri if resolved else "",
            "ancestor_paths": json.dumps(ancestry(graphs[release], URIRef(concept.iri))) if resolved else "[]"})
resolution = pd.DataFrame(resolution_rows)
resolution.to_csv(OUTPUT / "scenario_source_and_roots.csv", index=False)
display(resolution.head(8))

release,scenario_id,source_identifier,resolved,source_ontology,label,class_iri,ancestor_paths
before,P001,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1,True,HeTOP/EFMI/MIMO,Alert,http://example.org/hlo.owl#EFMI_0001_Alert,"[""EFMI_0001_Alert -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P002,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10,True,HeTOP/EFMI/MIMO,Biomedical Informatics,http://example.org/hlo.owl#EFMI_0002_Biomedical_Informatics,"[""EFMI_0002_Biomedical_Informatics -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P003,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10001,True,HeTOP/EFMI/MIMO,data display,http://example.org/hlo.owl#EFMI_0004_data_display,"[""EFMI_0004_data_display -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P004,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10003,True,HeTOP/EFMI/MIMO,computer-aided design,http://example.org/hlo.owl#EFMI_0006_computer_aided_design,"[""EFMI_0006_computer_aided_design -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P005,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10012,True,HeTOP/EFMI/MIMO,workflow,http://example.org/hlo.owl#EFMI_0014_workflow,"[""EFMI_0014_workflow -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P006,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10016,True,HeTOP/EFMI/MIMO,Agent-Based Modeling,http://example.org/hlo.owl#EFMI_0018_Agent_Based_Modeling,"[""EFMI_0018_Agent_Based_Modeling -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P007,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10021,True,HeTOP/EFMI/MIMO,probability theory,http://example.org/hlo.owl#EFMI_0023_probability_theory,"[""EFMI_0023_probability_theory -> MIMOConcept -> MedicalConcept -> Thing""]"
before,P008,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10023,True,HeTOP/EFMI/MIMO,library associations,http://example.org/hlo.owl#EFMI_0025_library_associations,"[""EFMI_0025_library_associations -> MIMOConcept -> MedicalConcept -> Thing""]"


## 9. Replay scenario selection without changing saved inputs

In [9]:
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

parent_module = load_module("audit_parent", EVALUATION / "generate_parent.py")
reference_module = load_module("audit_reference", EVALUATION / "generate_reference.py")
parent_rows = parent_module.generate(PATHS["before_owl"], PATHS["after_owl"], OUTPUT / "selected_parent_300.csv", count=300, seed=73)
reference_rows = reference_module.generate(PATHS["before_owl"], PATHS["after_owl"], OUTPUT / "selected_parent_300.csv", OUTPUT / "selected_reference_200.csv", count=200, seed=74)
selection = pd.DataFrame(parent_rows + reference_rows).fillna("")
assert set(selection.scenario_id) == set(structural.scenario_id)
selection_check = []
for scenario_type, regenerated in [("parent_placement", parent_rows), ("reference_lookup", reference_rows)]:
    saved = structural[structural.scenario_type.eq(scenario_type)]
    selection_check.append({"scenario_type": scenario_type, "count": len(regenerated),
                            "selected_source_ids_match": set(r["source_identifier"] for r in regenerated) == set(saved.source_identifier)})
display(pd.DataFrame(selection_check))
assert all(r["selected_source_ids_match"] for r in selection_check)
selection.to_csv(OUTPUT / "reselected_structural_500.csv", index=False)

scenario_type,count,selected_source_ids_match
parent_placement,300,True
reference_lookup,200,True


## 10. Selection limits

In [10]:
selection_limits = {
    "parent_seed": 73, "reference_seed": 74,
    "parent_pool": "MIMO identifiers shared by both releases with a label and before parent",
    "reference_pool": "shared MIMO identifiers excluding parent IDs; after must have a concept code",
    "reference_query": "after concept_code; this favors resolution in after",
    "expected_parent_source": "before OWL rather than an independent expert gold set",
    "seed_role": "scenario selection, not two recommendation trials",
}
(OUTPUT / "selection_protocol.json").write_text(json.dumps(selection_limits, indent=2), encoding="utf-8")
display(pd.DataFrame(selection_limits.items(), columns=["item", "value"]))

item,value
parent_seed,73
reference_seed,74
parent_pool,MIMO identifiers shared by both releases with a label and before parent
reference_pool,shared MIMO identifiers excluding parent IDs; after must have a concept code
reference_query,after concept_code; this favors resolution in after
expected_parent_source,before OWL rather than an independent expert gold set
seed_role,"scenario selection, not two recommendation trials"


## 11. Execute the structural evaluation

In [11]:
compare_module = load_module("audit_compare", EVALUATION / "compare_ontologies.py")
structural_before, structural_after, structural_comparison = compare_module.compare(
    PATHS["before_owl"], PATHS["after_owl"], PATHS["structural_scenarios"], OUTPUT / "structural_results")
display(pd.DataFrame(structural_comparison))
print("These are identifier/parent checks, not resource-ranking metrics.")

metric,before,after,difference
triples,14651.0,17839.0,3188.0
classes,1642.0,1642.0,0.0
source_identifier_values,1593.0,1593.0,0.0
concept_code_values,0.0,1610.0,1610.0
parent_cases,300.0,300.0,0.0
parent_passed,300.0,300.0,0.0
parent_accuracy,1.0,1.0,0.0
reference_cases,200.0,200.0,0.0
reference_passed,0.0,200.0,200.0
reference_accuracy,0.0,1.0,1.0


These are identifier/parent checks, not resource-ranking metrics.


## 12. Inspect every saved Excel trial

In [12]:
workbooks = {release: pd.read_excel(PATHS[f"{release}_excel"], sheet_name=0, keep_default_na=False)
             for release in ("before", "after")}
workbook_audit = []
for release, frame in workbooks.items():
    assert len(frame) == 500 and frame.scenario_id.is_unique
    assert set(frame.scenario_id) == set(structural.scenario_id)
    for metric in METRICS:
        trial_columns = [f"{metric}_trial1", f"{metric}_trial2"]
        values = frame[trial_columns].apply(pd.to_numeric)
        assert values.apply(lambda s: np.isfinite(s).all() and s.between(0, 1).all()).all()
        if metric in {"top1", "hit3", "hit5"}:
            assert values.isin([0, 1]).all().all()
        mean = values.mean(axis=1)
        workbook_audit.append({"release": release, "metric": metric,
            "rows": len(frame), "trial1_percent": values.iloc[:, 0].mean() * 100,
            "trial2_percent": values.iloc[:, 1].mean() * 100, "mean_percent": mean.mean() * 100,
            "stored_row_mean_matches": bool(np.allclose(mean, pd.to_numeric(frame[metric]))),
            "trial_rows_differ": int(values.iloc[:, 0].ne(values.iloc[:, 1]).sum())})
excel_audit = pd.DataFrame(workbook_audit)
excel_audit.to_csv(OUTPUT / "saved_excel_trial_audit.csv", index=False)
display(excel_audit)
print("Valid arithmetic does not identify the generator or establish original rankings.")

release,metric,rows,trial1_percent,trial2_percent,mean_percent,stored_row_mean_matches,trial_rows_differ
before,top1,500,61.800000,62.800000,62.3,True,229
before,hit3,500,75.200000,76.400000,75.8,True,182
before,hit5,500,82.800000,83.400000,83.1,True,143
before,mrr,500,69.033333,69.766667,69.4,True,281
before,map,500,70.510000,71.290000,70.9,True,283
after,top1,500,72.000000,76.400000,74.2,True,192
after,hit3,500,87.000000,85.200000,86.1,True,119
after,hit5,500,92.600000,92.000000,92.3,True,73
after,mrr,500,79.473333,81.126667,80.3,True,219
after,map,500,80.900000,82.100000,81.5,True,219


Valid arithmetic does not identify the generator or establish original rankings.


## 13. Basic rank consistency and Table 5 comparison

In [13]:
consistency = []
for release, frame in workbooks.items():
    for trial in (1, 2):
        top1, hit3, hit5, mrr = [pd.to_numeric(frame[f"{metric}_trial{trial}"]) for metric in ["top1", "hit3", "hit5", "mrr"]]
        consistency.append({"release": release, "trial": trial,
            "non_nested_hits": int(((top1 > hit3) | (hit3 > hit5)).sum()),
            "top1_mrr_conflicts": int((top1.eq(1) & ~np.isclose(mrr, 1)).sum()),
            "hit3_mrr_conflicts": int((hit3.eq(0) & mrr.gt(0.25 + 1e-12)).sum())})
display(pd.DataFrame(consistency))
table5 = pd.read_csv(PATHS["table5_summary"])
display(table5)
print("Original ranking logs, relevant resource sets, trial configurations and trial seeds are needed to explain Table 5.")

release,trial,non_nested_hits,top1_mrr_conflicts,hit3_mrr_conflicts
before,1,0,0,0
before,2,0,0,0
after,1,0,0,0
after,2,0,0,0


metric,baseline_percent,enriched_percent,point_gain,relative_gain_percent,operational_meaning,evaluation_scope
Top-1 accuracy,62.3,74.2,11.9,19.1,First-ranked item is relevant,500 rule-generated synthetic recommendation scenarios
Hit@3,75.8,86.1,10.3,13.6,At least one relevant item appears in the top three,500 rule-generated synthetic recommendation scenarios
Hit@5,83.1,92.3,9.2,11.1,At least one relevant item appears in the top five,500 rule-generated synthetic recommendation scenarios
MRR,69.4,80.3,10.9,15.7,Relevant items tend to occur early in the ranking,500 rule-generated synthetic recommendation scenarios
MAP,70.9,81.5,10.6,15.0,Relevant items are well ordered across the ranked list,500 rule-generated synthetic recommendation scenarios
PPA,NaN,89.7,NaN,NaN,Recommendation satisfies declared profile-policy constraints,Enriched HLO; baseline PPA not reported in Table 6


Original ranking logs, relevant resource sets, trial configurations and trial seeds are needed to explain Table 5.


## 14. The separate recommendation dataset

In [14]:
resources = pd.read_csv(PATHS["resources"])
context_scenarios = pd.read_csv(PATHS["recommendation_scenarios"])
context_concepts = pd.read_csv(PATHS["recommendation_concepts"])
assert len(context_scenarios) == 500
assert resources.resource_id.is_unique
assert set(resources.concept_id).issubset(set(context_concepts.concept_id))
context_scenarios = context_scenarios.reset_index(drop=True)
context_scenarios["audit_scenario_id"] = [f"C{i:03d}" for i in range(1, len(context_scenarios) + 1)]
display(pd.DataFrame([
    {"dataset": "structural", "rows": len(structural), "schema": " | ".join(structural.columns)},
    {"dataset": "recommendation", "rows": len(context_scenarios), "schema": " | ".join(context_scenarios.columns)},
    {"dataset": "resources", "rows": len(resources), "schema": " | ".join(resources.columns)},
]))
display(resources.head(5))
display(context_scenarios.head(5))

dataset,rows,schema
structural,500,scenario_id | scenario_type | source_identifier | query | expected_label | expected_category | expected_parent | selection_seed | expected_concept_code
recommendation,500,learner_id | level | concept_id | available_minutes | location | activity | audit_scenario_id
resources,600,resource_id | title | concept_id | min_level | max_level | duration_minutes | location | activity | quality | resource_type


resource_id,title,concept_id,min_level,max_level,duration_minutes,location,activity,quality,resource_type
HLO-R00001,Learning Resource 1,HLO-C0004,1,1,10,Clinic,ClinicalPractice,0.855,ClinicalGuideline
HLO-R00002,Learning Resource 2,HLO-C0076,1,1,5,Hospital,EducationalTask,0.655,ClinicalGuideline
HLO-R00003,Learning Resource 3,HLO-C0072,1,2,10,Home,Any,0.675,Video
HLO-R00004,Learning Resource 4,HLO-C0090,1,3,15,Clinic,EducationalTask,0.981,Article
HLO-R00005,Learning Resource 5,HLO-C0012,1,1,15,SimulationCenter,Any,0.669,Video


learner_id,level,concept_id,available_minutes,location,activity,audit_scenario_id
HLO-L0068,1,HLO-C0088,30,Home,EducationalTask,C001
HLO-L0036,5,HLO-C0087,60,Clinic,ResearchActivity,C002
HLO-L0041,3,HLO-C0037,60,Classroom,ResearchActivity,C003
HLO-L0088,4,HLO-C0081,45,Clinic,EducationalTask,C004
HLO-L0121,5,HLO-C0064,20,Clinic,ClinicalPractice,C005


## 15. OWL-to-resource identifier coverage

In [15]:
coverage_rows = []
resource_ids = set(resources.concept_id.astype(str))
for release, catalog in catalogs.items():
    owl_ids = set(catalog.class_local) | set(catalog.concept_code) | set(catalog.source_identifier) | set(catalog.iri)
    matched = sorted(resource_ids & owl_ids)
    coverage_rows.append({"release": release, "resource_concepts": len(resource_ids),
                          "direct_identifier_matches": len(matched), "matched_ids": json.dumps(matched)})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUTPUT / "owl_resource_identifier_coverage.csv", index=False)
display(coverage)
if coverage.direct_identifier_matches.eq(0).all():
    print("No direct identifier join connects the saved resource corpus to these two HLO OWL releases.")
print("An explicit, reviewed crosswalk is required before calling this an OWL before/after recommender experiment.")

release,resource_concepts,direct_identifier_matches,matched_ids
before,118,0,[]
after,118,0,[]


No direct identifier join connects the saved resource corpus to these two HLO OWL releases.
An explicit, reviewed crosswalk is required before calling this an OWL before/after recommender experiment.


## 16. Verify saved synthetic data generation

In [16]:
from hlo.data.generator import generate_dataset
config = yaml.safe_load(PATHS["config"].read_text())
dataset_seed = int(config["project"]["seed"])
data_config = config["data"]
regenerated = generate_dataset(n_concepts=int(data_config["n_concepts"]),
    n_resources=int(data_config["n_resources"]), n_learners=int(data_config["n_learners"]),
    n_scenarios=int(data_config["n_scenarios"]), seed=dataset_seed)
regeneration_checks = []
for name, saved in [("resources", resources), ("scenarios", context_scenarios.drop(columns="audit_scenario_id")), ("concepts", context_concepts)]:
    try:
        pd.testing.assert_frame_equal(saved, regenerated[name], check_dtype=False, atol=1e-12, rtol=0)
        match = True
    except AssertionError:
        match = False
    regeneration_checks.append({"dataset": name, "seed": dataset_seed, "saved_data_matches_generator": match})
display(pd.DataFrame(regeneration_checks))
print("This seed creates the supplied synthetic corpus. It is not a documented seed for Excel trial1 or trial2.")

dataset,seed,saved_data_matches_generator
resources,42,True
scenarios,42,True
concepts,42,True


This seed creates the supplied synthetic corpus. It is not a documented seed for Excel trial1 or trial2.


## 17. Resource selection and feature scoring

In [17]:
from hlo.recommendation.engine import RecommendationEngine, FEATURES
from hlo.evaluation.oracle import relevant_resource_ids
from hlo.evaluation.metrics import hit_at_k, reciprocal_rank, average_precision
saved_report = json.loads(PATHS["saved_summary"].read_text())
baseline_weights = config["recommendation"]["weights"]
trained_weights = saved_report["best_weights"]
constraints = config["recommendation"]["hard_constraints"]
engines = {
    "context_baseline": RecommendationEngine(weights=baseline_weights, **constraints),
    "context_trained": RecommendationEngine(weights=trained_weights, **constraints),
}
display(pd.DataFrame({"feature": FEATURES,
    "baseline_weight": [baseline_weights.get(f, 0) for f in FEATURES],
    "trained_weight": [trained_weights.get(f, 0) for f in FEATURES]}))
sample_scenario = context_scenarios.iloc[0].to_dict()
sample_ranked = engines["context_baseline"].rank(resources, sample_scenario, top_k=TOP_K)
print("Scenario:", sample_scenario)
print("Oracle relevant resources:", sorted(relevant_resource_ids(resources, sample_scenario)))
display(sample_ranked)
print("Reference relevance is a deterministic policy oracle, not an independent expert judgment.")

feature,baseline_weight,trained_weight
concept,0.05,0.246364
level,0.05,0.013318
duration,0.05,0.108149
location,0.10,0.088495
activity,0.10,0.283173
resource_quality,0.65,0.260502


resource_id,title,concept_id,min_level,max_level,duration_minutes,location,activity,quality,resource_type,score,feature_concept,feature_level,feature_duration,feature_location,feature_activity,feature_resource_quality
HLO-R00142,Learning Resource 142,HLO-C0089,1,1,20,Home,EducationalTask,0.935,Video,0.90775,0.0,1.0,1.0,1.0,1.0,0.935
HLO-R00275,Learning Resource 275,HLO-C0120,1,1,15,Home,EducationalTask,0.896,ClinicalGuideline,0.88240,0.0,1.0,1.0,1.0,1.0,0.896
HLO-R00144,Learning Resource 144,HLO-C0085,1,1,5,SimulationCenter,EducationalTask,0.995,ClinicalGuideline,0.84675,0.0,1.0,1.0,0.0,1.0,0.995
HLO-R00368,Learning Resource 368,HLO-C0079,1,2,15,Any,EducationalTask,0.833,Video,0.84145,0.0,1.0,1.0,1.0,1.0,0.833
HLO-R00076,Learning Resource 76,HLO-C0060,1,3,10,Classroom,Any,0.984,ClinicalGuideline,0.83960,0.0,1.0,1.0,0.0,1.0,0.984


Scenario: {'learner_id': 'HLO-L0068', 'level': 1, 'concept_id': 'HLO-C0088', 'available_minutes': 30, 'location': 'Home', 'activity': 'EducationalTask', 'audit_scenario_id': 'C001'}
Oracle relevant resources: ['HLO-R00452']
Reference relevance is a deterministic policy oracle, not an independent expert judgment.


## 18. Metric definitions

In [18]:
definitions = pd.DataFrame([
    ("top1", "1 if rank 1 is in the relevant set, otherwise 0"),
    ("hit3", "1 if any of ranks 1..3 is relevant, otherwise 0"),
    ("hit5", "1 if any of ranks 1..5 is relevant, otherwise 0"),
    ("mrr", "1 / rank of the first relevant item in the returned list; 0 if absent"),
    ("map", "Per-query AP@K: sum precision at relevant ranks / min(number relevant, number returned); then mean across queries"),
], columns=["metric", "project_definition"])
display(definitions)
print("The saved project AP denominator is retained here. Report TOP_K and this convention explicitly.")

metric,project_definition
top1,"1 if rank 1 is in the relevant set, otherwise 0"
hit3,"1 if any of ranks 1..3 is relevant, otherwise 0"
hit5,"1 if any of ranks 1..5 is relevant, otherwise 0"
mrr,1 / rank of the first relevant item in the returned list; 0 if absent
map,"Per-query AP@K: sum precision at relevant ranks / min(number relevant, number returned); then mean across queries"


The saved project AP denominator is retained here. Report TOP_K and this convention explicitly.


## 19. Execute actual rankings and preserve all evidence

In [19]:
rank_records, metric_records = [], []
for stage, engine in engines.items():
    for trial in (1, 2):
        for scenario in context_scenarios.to_dict("records"):
            relevant = relevant_resource_ids(resources, scenario)
            ranked = engine.rank(resources, scenario, top_k=TOP_K)
            ids = ranked.resource_id.astype(str).tolist()
            scores = ranked.score.astype(float).tolist()
            metrics = {"top1": hit_at_k(ids, relevant, 1), "hit3": hit_at_k(ids, relevant, 3),
                       "hit5": hit_at_k(ids, relevant, 5), "mrr": reciprocal_rank(ids, relevant),
                       "map": average_precision(ids, relevant)}
            rank_records.append({"stage": stage, "scenario_id": scenario["audit_scenario_id"],
                "trial": trial, "seed": None, "randomness": "none; deterministic repeat",
                "scenario": scenario, "ranked_resource_ids": ids, "scores": scores,
                "relevant_resource_ids": sorted(relevant), "weights": dict(engine.weights),
                "top_k": TOP_K, "ontology_release": None,
                "dataset_sha256": file_hash(PATHS["recommendation_scenarios"]), "metrics": metrics})
            metric_records.append({"stage": stage, "scenario_id": scenario["audit_scenario_id"], "trial": trial, **metrics})
with (OUTPUT / "new_context_rankings.jsonl").open("w", encoding="utf-8") as stream:
    for record in rank_records:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
new_metrics = pd.DataFrame(metric_records)
new_metrics.to_csv(OUTPUT / "new_context_trial_metrics.csv", index=False)
print("Ranking records:", len(rank_records))
display(new_metrics.head())

stage,scenario_id,trial,top1,hit3,hit5,mrr,map
context_baseline,C001,1,0.0,0.0,0.0,0.000000,0.000000
context_baseline,C002,1,0.0,0.0,1.0,0.200000,0.200000
context_baseline,C003,1,0.0,1.0,1.0,0.333333,0.166667
context_baseline,C004,1,1.0,1.0,1.0,1.000000,1.000000
context_baseline,C005,1,0.0,0.0,0.0,0.000000,0.000000


Ranking records: 2000


## 20. Deterministic trial checks and new results

In [20]:
for stage in engines:
    selected = new_metrics[new_metrics.stage.eq(stage)]
    a = selected[selected.trial.eq(1)].set_index("scenario_id")[METRICS]
    b = selected[selected.trial.eq(2)].set_index("scenario_id")[METRICS]
    pd.testing.assert_frame_equal(a, b)
new_summary = new_metrics.groupby("stage")[METRICS].mean().mul(100).reset_index()
new_summary.to_csv(OUTPUT / "new_context_summary.csv", index=False)
display(new_summary)
print("Both trials are deterministic repeats; no independent random trial seed is introduced.")
print("These are fresh results on context scenarios C001..C500, not Table 5 or the P/R trial values.")

stage,top1,hit3,hit5,mrr,map
context_baseline,22.0,33.0,45.8,29.323333,25.945
context_trained,99.8,100.0,100.0,99.866667,99.800


Both trials are deterministic repeats; no independent random trial seed is introduced.
These are fresh results on context scenarios C001..C500, not Table 5 or the P/R trial values.


## 21. Export the new measured Excel results

In [21]:
for stage in engines:
    subset = new_metrics[new_metrics.stage.eq(stage)]
    details = context_scenarios.copy()
    for metric in METRICS:
        pivot = subset.pivot(index="scenario_id", columns="trial", values=metric)
        for trial in (1, 2):
            details[f"{metric}_trial{trial}"] = details.audit_scenario_id.map(pivot[trial])
        details[metric] = details[[f"{metric}_trial1", f"{metric}_trial2"]].mean(axis=1)
    destination = OUTPUT / f"NEW_{stage}_500.xlsx"
    with pd.ExcelWriter(destination, engine="openpyxl") as writer:
        details.to_excel(writer, sheet_name="scenarios", index=False)
        new_summary[new_summary.stage.eq(stage)].to_excel(writer, sheet_name="summary", index=False)
        pd.DataFrame([{"scope": "new deterministic context ranking run", "historical_table5": False,
                       "original_excel_reproduced": False, "trial_seeds": "not applicable", "top_k": TOP_K}]).to_excel(writer, sheet_name="provenance", index=False)
        for sheet in writer.book.worksheets:
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
    print(destination.name)

NEW_context_baseline_500.xlsx
NEW_context_trained_500.xlsx


## 22. Contract for historical recommendation evidence

In [22]:
original_log_contract = {
    "format": "one JSON object per line",
    "required_fields": ["release", "scenario_id", "trial", "seed", "ontology_sha256",
                        "resources_sha256", "top_k", "ranked_resource_ids", "relevant_resource_ids"],
    "release_values": ["before", "after"], "trial_values": [1, 2],
    "expected_records": 2000, "scenario_set": "exact saved P001..P300 and R001..R200",
    "seed": "integer with a recorded role, or null for a deterministic run",
    "additional_required_provenance": ["original runner path/hash", "trial configuration", "crosswalk and ground-truth provenance"],
}
(OUTPUT / "historical_log_contract.json").write_text(json.dumps(original_log_contract, indent=2), encoding="utf-8")
display(pd.DataFrame(original_log_contract.items(), columns=["item", "value"]))
print("No fabricated template ranking records are exported.")

item,value
format,one JSON object per line
required_fields,"[release, scenario_id, trial, seed, ontology_sha256, resources_sha256, top_k, ranked_resource_ids, relevant_resource_ids]"
release_values,"[before, after]"
trial_values,"[1, 2]"
expected_records,2000
scenario_set,exact saved P001..P300 and R001..R200
seed,"integer with a recorded role, or null for a deterministic run"
additional_required_provenance,"[original runner path/hash, trial configuration, crosswalk and ground-truth provenance]"


No fabricated template ranking records are exported.


## 23. Recompute trial values from original ranked lists if supplied

In [23]:
historical_replay_status = "blocked_original_rankings_not_supplied"
historical_comparison = None
if RAW_ORIGINAL_LOG is not None:
    path = Path(RAW_ORIGINAL_LOG)
    log = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    required = set(original_log_contract["required_fields"])
    expected = {(release, sid, trial) for release in ("before", "after") for sid in structural.scenario_id for trial in (1, 2)}
    keys, measured = [], []
    for row in log:
        if not required.issubset(row):
            raise ValueError(f"Missing log fields: {required - set(row)}")
        key = (row["release"], row["scenario_id"], row["trial"])
        keys.append(key)
        assert row["ontology_sha256"] == file_hash(PATHS[f"{row['release']}_owl"])
        assert row["resources_sha256"] == file_hash(PATHS["resources"])
        assert row["seed"] is None or (isinstance(row["seed"], int) and not isinstance(row["seed"], bool))
        ranked, relevant = row["ranked_resource_ids"], set(row["relevant_resource_ids"])
        assert len(ranked) == len(set(ranked)) and len(ranked) <= row["top_k"]
        assert set(ranked).issubset(set(resources.resource_id.astype(str)))
        assert relevant.issubset(set(resources.resource_id.astype(str)))
        measured.append({"release": key[0], "scenario_id": key[1], "trial": key[2],
            "top1": hit_at_k(ranked, relevant, 1), "hit3": hit_at_k(ranked, relevant, 3),
            "hit5": hit_at_k(ranked, relevant, 5), "mrr": reciprocal_rank(ranked, relevant),
            "map": average_precision(ranked, relevant)})
    assert set(keys) == expected and len(keys) == len(expected)
    historical = pd.DataFrame(measured)
    checks = []
    for release, excel in workbooks.items():
        for trial in (1, 2):
            actual = historical[historical.release.eq(release) & historical.trial.eq(trial)].set_index("scenario_id")
            stored = excel.set_index("scenario_id")
            for metric in METRICS:
                diff = actual[metric].reindex(stored.index) - pd.to_numeric(stored[f"{metric}_trial{trial}"])
                checks.append({"release": release, "trial": trial, "metric": metric,
                               "mismatched_rows": int((diff.abs() > 1e-12).sum()), "max_absolute_error": float(diff.abs().max())})
    historical_comparison = pd.DataFrame(checks)
    historical_comparison.to_csv(OUTPUT / "historical_rank_excel_comparison.csv", index=False)
    display(historical_comparison)
    historical_replay_status = "rank_metrics_match_supplied_excel" if historical_comparison.mismatched_rows.eq(0).all() else "rank_metrics_do_not_match_supplied_excel"
else:
    print(historical_replay_status)

blocked_original_rankings_not_supplied


## 24. Publication evidence checklist

In [24]:
checklist = pd.DataFrame([
    ("Saved 300+200 scenarios", True, "evaluation/data/scenarios.csv"),
    ("Scenario selection reproducible", all(r["selected_source_ids_match"] for r in selection_check), "selection seeds 73 and 74"),
    ("Structural checks executable", True, "compare_ontologies.py"),
    ("Separate recommendation engine executable", True, "context dataset; saved weights; new raw rankings"),
    ("OWL-resource crosswalk for Table 5", False, "not recovered from supplied files"),
    ("Original P/R resource-ranked lists", RAW_ORIGINAL_LOG is not None, historical_replay_status),
    ("Historical trial1/trial2 seed protocol", False, "not established by selection or dataset seeds"),
    ("Original Table 5 runner identified", False, "no verified filename recovered"),
    ("Target-matched examples suitable as original experimental evidence", False, "previously constructed to match aggregate targets"),
], columns=["requirement", "available_or_verified", "evidence"])
checklist.to_csv(OUTPUT / "publication_evidence_checklist.csv", index=False)
display(checklist)

requirement,available_or_verified,evidence
Saved 300+200 scenarios,True,evaluation/data/scenarios.csv
Scenario selection reproducible,True,selection seeds 73 and 74
Structural checks executable,True,compare_ontologies.py
Separate recommendation engine executable,True,context dataset; saved weights; new raw rankings
OWL-resource crosswalk for Table 5,False,not recovered from supplied files
Original P/R resource-ranked lists,False,blocked_original_rankings_not_supplied
Historical trial1/trial2 seed protocol,False,not established by selection or dataset seeds
Original Table 5 runner identified,False,no verified filename recovered
Target-matched examples suitable as original experimental evidence,False,previously constructed to match aggregate targets


## 25. Export audit manifest and output inventory

In [25]:
output_inventory = pd.DataFrame([
    {"file": str(path.relative_to(OUTPUT)), "bytes": path.stat().st_size, "sha256": file_hash(path)}
    for path in sorted(OUTPUT.rglob("*")) if path.is_file() and path.name != "audit_manifest.json"
])
output_inventory.to_csv(OUTPUT / "output_inventory.csv", index=False)
manifest = {
    "operation": "source audit, structural replay and new context recommendation execution",
    "historical_table5_reproduced": False,
    "original_table5_runner_recovered": False,
    "prior_excel_provenance": "target-matched illustrative values in prior creation context; original ranking execution not recovered",
    "structural_cases": 500, "context_cases": 500, "resource_count": len(resources),
    "selection_seeds": {"parent": 73, "reference": 74}, "dataset_seed": dataset_seed,
    "new_ranking_trial_seeds": [None, None], "new_trials": "deterministic repeats",
    "historical_rank_log_status": historical_replay_status,
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "rdflib": rdflib.__version__,
    "input_files": inventory.to_dict("records"), "new_context_summary": new_summary.to_dict("records"),
    "output_files": output_inventory.to_dict("records"),
}
(OUTPUT / "audit_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(output_inventory[["file", "bytes"]])
print("Completed. Outputs are under evaluation/execution_audit_500.")

file,bytes
NEW_context_baseline_500.xlsx,43225
NEW_context_trained_500.xlsx,41419
after_ontology_catalog.csv,284524
before_ontology_catalog.csv,368558
code_roles.csv,1173
historical_log_contract.json,644
input_inventory.csv,1709
new_context_rankings.jsonl,1843318
new_context_summary.csv,157
new_context_trial_metrics.csv,89395


Completed. Outputs are under evaluation/execution_audit_500.
